# Mini-Project about sentiment analysis

The purpose of this project is to conduct the sentiment analysis of movie reviews. The data comes from Rotten Tomatoes service and it was originally classified into 5 classes: negative, somewhat negative, neutral, somewhat positive and postive. In this notebook model classifying the reviews is going to be created.

author: Alicja Martinek

### load dependencies

In [63]:
import numpy as np
import pandas as pd
import gensim

import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, SpatialDropout1D, Embedding, LSTM
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

np.random.seed(42)

import matplotlib.pyplot as plt
%matplotlib inline

### load data

In [38]:
train_data = pd.read_csv('train.tsv', sep='\t', header = 0)
test_data = pd.read_csv('test.tsv', sep='\t', header = 0)

### data preprocessing

Creating own word embeddings the way it's done below results in no semantic similarity.

In [56]:
words_amount = 2000

#split into words
tokenizer = Tokenizer(num_words = words_amount, split=' ')
tokenizer.fit_on_texts(train_data['Phrase'].values)

#change into numerical vals
train_data_num = tokenizer.texts_to_sequences(train_data['Phrase'].values)
test_data_num = tokenizer.texts_to_sequences(test_data['Phrase'].values)

#reshape data
train_data_pad = pad_sequences(train_data_num)
test_data_pad = pad_sequences(test_data_num)

#one hot
train_labels = pd.get_dummies(train_data['Sentiment']).values

#showing how labels look, seems legit because there are five categories
print(train_labels[0:5])

[[0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 1 0 0]]


### set hyperparameters

In [58]:
embedding_dimension = 128
lstm_out = 128
batch_size = 128
drop = 0.3
classes_num = 5

### model architecture

In [59]:
model = Sequential()
model.add(Embedding(words_amount, embedding_dimension, input_length=train_data_pad.shape[1]))
model.add(SpatialDropout1D(0.6))
model.add(LSTM(lstm_out))
model.add(Dropout(drop))
model.add(Dense(classes_num, activation='softmax'))

### model parameters

In [60]:
model.compile(loss='categorical_crossentropy', optimizer = 'adam', metrics =['accuracy'])
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_6 (Embedding)      (None, 46, 128)           256000    
_________________________________________________________________
spatial_dropout1d_6 (Spatial (None, 46, 128)           0         
_________________________________________________________________
lstm_6 (LSTM)                (None, 128)               131584    
_________________________________________________________________
dropout_6 (Dropout)          (None, 128)               0         
_________________________________________________________________
dense_6 (Dense)              (None, 5)                 645       
Total params: 388,229
Trainable params: 388,229
Non-trainable params: 0
_________________________________________________________________


### training!

In [61]:
model.fit(train_data_pad, train_labels, epochs=4, batch_size=batch_size, verbose=1)

Epoch 1/4
156060/156060 [==============================] - 590s 4ms/step - loss: 1.0800 - acc: 0.5741
Epoch 2/4
156060/156060 [==============================] - 494s 3ms/step - loss: 0.9777 - acc: 0.6143
Epoch 3/4
156060/156060 [==============================] - 494s 3ms/step - loss: 0.9532 - acc: 0.6245
Epoch 4/4
156060/156060 [==============================] - 515s 3ms/step - loss: 0.9360 - acc: 0.6300


### predictions - just to show that it works

In [ ]:
y_hat = model.predict_classes(test_data_pad)

### comment 

Created model had 63% accuracy on training set. As labels for test set were not provided, accuracy on test set couldn't be evaluated.
No fitting of hyperparameters was performed, due to the time of learnig (time per epoch) and the fact that lecturer said that the project should be a 30 minutes task.